# Part 2: Flow Matching Parameterization

In [4]:
from src.model import JiM
from src.train import train_one_epoch
from src.denoiser import Denoiser
from src.dataloader import get_dataloader

import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
from pathlib import Path

In [9]:
AVAILABLE_DATASETS = ("swiss_roll", "gaussians", "circles")
DIMS = (2, 8, 32)
PRED_LOSS = (("x", "x"), ("x", "v"), ("v", "x"), ("v", "v"))

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 5000          # adjust to hit ~25k steps given your dataset size
STEPS = 50
BATCH = 1024
N_GEN = 2048           # samples to generate for visualization

### Helpers

In [10]:
def model_tag(dataset, D, pred, loss):
    return f"{dataset}_D{D}_{pred}pred_{loss}loss"


def train_one_config(dataset, D, pred_type, loss_type, device):
    dataloader = get_dataloader(name=dataset, dim=D, batch_size=BATCH)
    model = JiM(hidden_dim=256, D=D).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    for ep in range(EPOCHS):
        train_one_epoch(model, dataloader, opt, device, ep, 500, pred_type, loss_type)

    path = MODEL_DIR / f"{model_tag(dataset, D, pred_type, loss_type)}.pt"
    torch.save(model.state_dict(), path)
    return model


def generate_samples(model, D, pred_type, device, n=N_GEN):
    denoiser = Denoiser(model, steps=STEPS, D=D)
    return denoiser.generate(n, pred_type, device).cpu().numpy()


def get_ground_truth(dataset, D, n=N_GEN):
    loader = get_dataloader(name=dataset, dim=D, batch_size=n)
    return next(iter(loader))[:n].cpu().numpy()


def project_to_2d(samples, dataset, D):
    if D == 2:
        return samples
    loader = get_dataloader(name=dataset, dim=D, batch_size=1)
    return loader.dataset.to_2d(torch.from_numpy(samples)).numpy()


def plot_grid(dataset, device):
    """One figure per dataset: 4 rows (pred/loss combos) × 3 cols (D ∈ {2,8,32}).
    Each cell overlays generated samples (color) over ground truth (light gray)."""
    fig, axes = plt.subplots(
        len(PRED_LOSS), len(DIMS),
        figsize=(4 * len(DIMS), 4 * len(PRED_LOSS)),
        squeeze=False,
    )

    for row, (pred, loss) in enumerate(PRED_LOSS):
        for col, D in enumerate(DIMS):
            ax = axes[row, col]
            tag = model_tag(dataset, D, pred, loss)

            model = JiM(hidden_dim=256, D=D).to(device)
            model.load_state_dict(torch.load(MODEL_DIR / f"{tag}.pt", map_location=device))
            model.eval()

            gen = generate_samples(model, D, pred, device)
            gt = get_ground_truth(dataset, D)

            gen2d = project_to_2d(gen, dataset, D)
            gt2d = project_to_2d(gt, dataset, D)

            ax.scatter(gt2d[:, 0],  gt2d[:, 1],  s=3, alpha=0.25, c="lightgray", label="ground truth")
            ax.scatter(gen2d[:, 0], gen2d[:, 1], s=3, alpha=0.55, c="C0",        label="generated")
            ax.set_title(f"{pred}-pred + {loss}-loss   D={D}", fontsize=11)
            ax.set_aspect("equal")
            ax.set_xticks([]); ax.set_yticks([])

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.02), fontsize=11)
    fig.suptitle(f"Dataset: {dataset}", fontsize=14, y=1.05)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{dataset}_grid.png", dpi=150, bbox_inches="tight")
    plt.show()

### Training Loop

In [ ]:
# === Train all 36 configs ===
device = "cuda" if torch.cuda.is_available() else "cpu"

for dataset in AVAILABLE_DATASETS:
    for D in DIMS:
        for pred, loss in PRED_LOSS:
            tag = model_tag(dataset, D, pred, loss)
            path = MODEL_DIR / f"{tag}.pt"
            if path.exists():
                print(f"[skip] {tag} already trained")
                continue
            print(f"[train] {tag}")
            train_one_config(dataset, D, pred, loss, device)

# === Plot one grid per dataset ===
for dataset in AVAILABLE_DATASETS:
    plot_grid(dataset, device)

[train] swiss_roll_D2_xpred_xloss
